# Stage the Ignite 2026 Zava workshop
Import this notebook into a blank Fabric workspace or put it in your deployment folder. Use active capacity and do not attach a default Lakehouse.
The notebook downloads public JSON/TMDL files from the workshop GitHub repository. It never downloads or executes remote Python. HTTPS access to api.github.com and raw.githubusercontent.com is required; no GitHub credentials or Git installation needed.
Run all with APPLY=False to preview. Confirm the workspace name, set APPLY=True, then run all.
The Lakehouse has 19 tables; six dependency entities are deliberately absent from the unchanged 13-table semantic model. No ontology is prebuilt.
REPO_REVISION can be a commit, tag or branch. Branches/tags resolve once to a commit per run. Pin a full commit for a repeatable workshop. See README.md in the repository.


In [ ]:
APPLY = False
CONFIRM_WORKSPACE_NAME = ""
REPO_REVISION = 'main'


## Deployment runtime
Reviewed local code; source-file downloads never receive Fabric tokens. Existing ownership, folder and no-overwrite guards remain.


In [ ]:
"""Reviewed Fabric PySpark installer; public data downloads are never executed."""
import base64
from datetime import datetime, timezone
import hashlib
import io
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from uuid import UUID
import zipfile

FABRIC_API = "https://api.fabric.microsoft.com"
POWERBI_API = "https://api.powerbi.com"
SOURCE_REPOSITORY = "adamlash/ontology-ignite-2026"
RAW_ROOT = "https://raw.githubusercontent.com/" + SOURCE_REPOSITORY + "/"
LEGACY_BASE_CONTENT = "dd24de81bf98363784c8c9783c6ab802ba117ffdb03d86d1f317a585c61eb987"
LEGACY_BUNDLE_IDENTITY = "679c45aa6d06a8661ebaa181fe5d03cb8102b13e930ea1c44dad9612dfc9461c"


def check(condition, message):
    if not condition:
        raise RuntimeError(message)


def uuid_text(value):
    return str(UUID(value))


def timestamp():
    return datetime.now(timezone.utc).isoformat()


def unpack_assets(encoded, digest):
    raw = base64.b64decode(encoded, validate=True)
    check(hashlib.sha256(raw).hexdigest() == digest, "Embedded input archive checksum mismatch")
    files = {}
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        check(sum(i.file_size for i in archive.infolist()) < 20_000_000, "Input archive too large")
        for info in archive.infolist():
            check(not info.is_dir() and not info.filename.startswith(("/", "\\"))
                  and ".." not in info.filename.split("/") and "\\" not in info.filename,
                  "Unexpected input file path")
            check(info.filename not in files, "Duplicate archive path")
            files[info.filename] = archive.read(info).decode("utf-8")
    return files


class DenyRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        raise RuntimeError("HTTP redirect refused; credentials were not forwarded")


def public_download(url, limit):
    parsed = urllib.parse.urlsplit(url)
    check(parsed.scheme == "https" and parsed.port in (None, 443)
          and not parsed.username and not parsed.password and not parsed.query and not parsed.fragment
          and (url.startswith(RAW_ROOT) or url.startswith(
              "https://api.github.com/repos/" + SOURCE_REPOSITORY + "/commits/")),
          "Unexpected public workshop source URL")
    headers = {"User-Agent": "Ontology-Ignite-Workshop"}
    if parsed.hostname == "api.github.com":
        headers["Accept"] = "application/vnd.github.sha"
    request = urllib.request.Request(url, headers=headers)
    opener = urllib.request.build_opener(DenyRedirect())
    for attempt in range(3):
        try:
            with opener.open(request, timeout=60) as response:
                check(response.status == 200, "Unexpected public source response")
                raw = response.read(limit + 1)
                check(len(raw) <= limit, "Public input exceeds download size limit")
                return raw
        except urllib.error.HTTPError as error:
            status = error.code
            retry_after = error.headers.get("Retry-After", "") if error.headers else ""
            error.close()
            if status in (429, 500, 502, 503, 504) and attempt < 2:
                delay = max(5 * 2 ** attempt, int(retry_after) if retry_after.isdigit() else 0)
                check(delay <= 120, "Public source Retry-After exceeds automatic wait; try again later")
                print("Public source HTTP", status, "- retrying read in", delay, "seconds")
                time.sleep(delay)
                continue
            raise RuntimeError(f"Public source HTTP {status}: {parsed.path}; no inputs were ingested") from None


def content_digest(entries):
    hashes = {name: hashlib.sha256(raw).hexdigest() for name, raw in entries.items()}
    return hashlib.sha256(json.dumps(hashes, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def download_inputs(revision="main"):
    check(isinstance(revision, str) and re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]{0,199}", revision)
          and ".." not in revision and "//" not in revision, "Invalid repository revision")
    if re.fullmatch(r"[0-9a-f]{40}", revision):
        commit = revision
    else:
        commit = public_download(
            "https://api.github.com/repos/" + SOURCE_REPOSITORY + "/commits/"
            + urllib.parse.quote(revision, safe=""), 1000).decode("ascii").strip()
    check(isinstance(commit, str) and re.fullmatch(r"[0-9a-f]{40}", commit), "Invalid resolved commit")
    root = RAW_ROOT + commit + "/"
    manifest_raw = public_download(root + "manifest.json", 100_000)
    manifest = json.loads(manifest_raw)
    check(manifest["version"] == 1 and isinstance(manifest["files"], list)
          and 1 <= len(manifest["files"]) <= 100, "Unsupported input manifest")
    files, base_entries, paths = {}, {}, set()
    total = 0
    for entry in manifest["files"]:
        name, path = entry["name"], entry["path"]
        check(isinstance(name, str) and re.fullmatch(r"[A-Za-z0-9_./-]+", name)
              and all(part not in ("", ".", "..") for part in name.split("/"))
              and name.endswith((".json", ".tmdl", ".pbism")), "Unsafe input manifest name")
        expected = public_input_path(name)
        check(path == expected and name not in files and path not in paths,
              "Duplicate or unexpected input manifest path")
        size = entry["bytes"]
        check(type(size) is int and 0 < size <= 2_000_000
              and re.fullmatch(r"[0-9a-f]{64}", entry["sha256"]), "Invalid input checksum/size")
        total += size
        check(total <= 20_000_000, "Manifest inputs exceed total size limit")
        raw = public_download(root + path, size)
        check(len(raw) == size and hashlib.sha256(raw).hexdigest() == entry["sha256"],
              "Downloaded input checksum/size mismatch: " + name)
        files[name] = raw.decode("utf-8")
        paths.add(path)
        if not name.startswith("dependencies/"):
            base_entries[name] = raw
    fingerprint = content_digest(base_entries)
    check(fingerprint == manifest["baseContentSha256"], "Downloaded base content identity mismatch")
    # Preserve legacy ownership only for these exact known file bytes, never for changed data.
    digest = LEGACY_BUNDLE_IDENTITY if fingerprint == LEGACY_BASE_CONTENT else fingerprint
    check(digest == manifest["bundleSha256"], "Downloaded base bundle identity mismatch")
    contract, schemas = validate_inputs(files)
    dependencies, _ = validate_dependencies(files, contract, schemas)
    check(len(contract["tables"]) + len(dependencies.get("tables", {})) == manifest["lakehouseTables"],
          "Manifest table count mismatch")
    provenance = {"repository": SOURCE_REPOSITORY, "commit": commit,
                  "manifestSha256": hashlib.sha256(manifest_raw).hexdigest()}
    print("Verified public workshop inputs:", len(files), "files at commit", commit)
    return files, digest, provenance


def public_input_path(name):
    if name.startswith("model/"):
        return name
    if name.startswith("data/"):
        return "data/tables/" + name[len("data/"):]
    return "data/" + name


def fabric_operation_url(location, operation_id=None):
    if location:
        parsed = urllib.parse.urlsplit(location)
        host = parsed.hostname or ""
        check(parsed.scheme == "https" and parsed.port in (None, 443)
              and not parsed.username and not parsed.password and not parsed.query and not parsed.fragment
              and (host == "api.fabric.microsoft.com"
                   or host.endswith((".analysis.windows.net", ".msitapi.fabric.microsoft.com"))),
              "Unexpected asynchronous operation URL")
        match = re.fullmatch(r"/v1/operations/([a-fA-F0-9-]{36})(?:/result)?", parsed.path)
        check(match, "Unexpected asynchronous operation path")
        identifier = uuid_text(match[1])
        check(not operation_id or uuid_text(operation_id) == identifier,
              "Asynchronous operation identifiers disagree")
    else:
        check(operation_id, "No operation Location or ID; inspect before retrying")
        identifier = uuid_text(operation_id)
    # Regional service hosts can issue the Location; tokens still go only to the public Fabric API.
    return FABRIC_API + "/v1/operations/" + identifier


class FabricClient:
    def __init__(self, utilities):
        self.utilities = utilities
        self.opener = urllib.request.build_opener(DenyRedirect())

    def call(self, method, url, body=None, audience="pbi"):
        parsed = urllib.parse.urlsplit(url)
        check(parsed.scheme == "https" and parsed.port in (None, 443)
              and not parsed.username and not parsed.password, "Unsafe endpoint")
        host = parsed.hostname or ""
        check((audience == "pbi" and host in ("api.fabric.microsoft.com", "api.powerbi.com"))
              or (audience == "kusto" and host.endswith(".kusto.fabric.microsoft.com")),
              "Refusing to send a token to this host/audience")
        request = urllib.request.Request(
            url, method=method, data=json.dumps(body).encode() if body is not None else None,
            headers={"Content-Type": "application/json",
                     "Authorization": "Bearer " + self.utilities.credentials.getToken(audience)})
        attempts = 3 if method == "GET" else 1
        for attempt in range(attempts):
            try:
                with self.opener.open(request, timeout=120) as response:
                    raw = response.read()
                    return response.status, {k.lower(): v for k, v in response.headers.items()}, (
                        json.loads(raw) if raw else {})
            except urllib.error.HTTPError as error:
                status = error.code
                retry_after = error.headers.get("Retry-After", "") if error.headers else ""
                error.close()
                if method == "GET" and status in (429, 500, 502, 503, 504) and attempt + 1 < attempts:
                    delay = max(5 * 2 ** attempt, int(retry_after) if retry_after.isdigit() else 0)
                    check(delay <= 120,
                          f"HTTP {status}: GET {parsed.path}. Retry-After exceeds automatic wait; resume later.")
                    print(f"HTTP {status}: GET {parsed.path}; retrying read in {delay}s "
                          f"({attempt + 2}/{attempts}).")
                    time.sleep(delay)
                    continue
                note = f"Read failed after {attempt + 1} attempt(s)." if method == "GET" else "Not retried."
                raise RuntimeError(f"HTTP {status}: {method} {parsed.path}. {note}") from None

    def get(self, url):
        status, _, body = self.call("GET", url)
        check(status == 200, "Unexpected read response")
        return body

    def listing(self, url):
        result, seen = [], set()
        token = None
        while True:
            query = "?" + urllib.parse.urlencode({"continuationToken": token}) if token else ""
            target = url + query
            check(target not in seen and len(seen) < 100, "Pagination loop or excessive results")
            seen.add(target)
            page = self.get(target)
            result.extend(page["value"])
            token = page.get("continuationToken")
            check(not page.get("continuationUri") or token, "Unsupported URI-only continuation")
            if not token:
                return result

    def finish_operation(self, location, timeout=900, operation_id=None):
        operation_url = fabric_operation_url(location, operation_id)
        deadline = time.monotonic() + timeout
        while time.monotonic() < deadline:
            body = self.get(operation_url)
            if body["status"] == "Succeeded":
                return self.get(operation_url + "/result")
            check(body["status"] not in ("Failed", "Canceled", "Cancelled"), "Fabric operation failed")
            time.sleep(10)
        raise TimeoutError("Fabric operation still pending. Inspect the saved operation; do not re-create.")

    def create(self, url, body, before_poll=None):
        status, headers, result = self.call("POST", url, body)
        if status == 202:
            location = headers.get("location", "")
            operation_id = headers.get("x-ms-operation-id")
            check(location or operation_id, "No operation Location or ID; inspect before retrying")
            if before_poll:
                before_poll(location or fabric_operation_url("", operation_id))
            return self.finish_operation(location, operation_id=operation_id)
        check(status == 201, "Unexpected create response")
        return result


def one_lake_files(properties, workspace_id, lakehouse_id):
    parsed = urllib.parse.urlsplit(properties["oneLakeFilesPath"])
    check(parsed.scheme == "https" and parsed.port in (None, 443)
          and not parsed.username and not parsed.password and not parsed.query and not parsed.fragment,
          "Unexpected OneLake path")
    check(parsed.hostname in ("onelake.dfs.fabric.microsoft.com", "msit-onelake.dfs.fabric.microsoft.com",
                              "dxt-onelake.dfs.fabric.microsoft.com"), "Unexpected OneLake host")
    expected = f"/{workspace_id}/{lakehouse_id}/Files"
    check(parsed.path.rstrip("/") == expected, "OneLake path is not this workspace/lakehouse")
    return f"abfss://{workspace_id}@{parsed.hostname}/{lakehouse_id}/Files", (
        f"https://{parsed.hostname}/{workspace_id}/{lakehouse_id}")


def render_model(files, one_lake_root):
    check(one_lake_root.startswith("https://") and '"' not in one_lake_root, "Invalid model root")
    result = []
    for name in sorted(files):
        if name.startswith("model/"):
            text = files[name].replace("{{ONELAKE_ROOT}}", one_lake_root)
            check("{{" not in text, "Unresolved model parameter")
            check("lineageTag:" not in text and not name.endswith(".platform"), "Nonportable model metadata")
            result.append({"path": name[len("model/"):], "payloadType": "InlineBase64",
                           "payload": base64.b64encode(text.encode()).decode()})
    check(result, "No semantic model definition")
    return {"format": "TMDL", "parts": result}


def kql_result(body):
    check(not body.get("error") and body.get("Tables"), "KQL returned no usable response")
    for table in body["Tables"]:
        columns = [c["ColumnName"] for c in table["Columns"]]
        if "Severity" in columns:
            check(all(r[columns.index("Severity")] > 2 for r in table["Rows"]),
                  "KQL command/query reported a failure or partial failure")
    primary = [t for t in body["Tables"] if t["TableName"] == "Table_0"]
    check(len(primary) == 1, "KQL primary result is missing or ambiguous")
    table = primary[0]
    columns = [c["ColumnName"] for c in table["Columns"]]
    return [dict(zip(columns, row)) for row in table["Rows"]]


def quote_identifier(value):
    return "`" + value.replace("`", "``") + "`"


def table_identifier(workspace_name, lakehouse_name, physical):
    check(re.fullmatch(r"[a-z]+", physical), "Unsafe physical table identifier")
    return ".".join(map(quote_identifier, (workspace_name, lakehouse_name, "dbo", physical)))


def validate_inputs(files):
    contract = json.loads(files["contract.json"])
    schemas = json.loads(files["schemas.json"])
    check(set(contract["tables"]) == set(schemas), "Table/schema mismatch")
    for name, spec in contract["tables"].items():
        rows = json.loads(files["data/" + name + ".json"])
        check(len(rows) == spec["rows"], "Fixture row-count mismatch: " + name)
        check(all(set(row) == set(schemas[name]) for row in rows), "Fixture column mismatch: " + name)
        keys = [row[spec["key"]] for row in rows]
        check(None not in keys and "" not in keys and len(set(keys)) == len(keys),
              "Fixture keys missing or duplicated: " + name)
    return contract, schemas


def validate_dependencies(files, base_contract, base_schemas):
    if not any(name.startswith("dependencies/") for name in files):
        return {}, {}
    contract = json.loads(files["dependencies/contract.json"])
    schemas = json.loads(files["dependencies/schemas.json"])
    tables = contract["tables"]
    check(contract["version"] == 1 and contract["asOfUtc"] == base_contract["asOfUtc"],
          "Unsupported dependency version/checkpoint")
    check(set(tables) == set(schemas) and not set(tables).intersection(base_contract["tables"]),
          "Dependency entities must be new Lakehouse-only tables")
    all_specs = {**base_contract["tables"], **tables}
    physical = [spec["physical"] for spec in all_specs.values()]
    check(all(re.fullmatch(r"[a-z]+", name) for name in physical) and len(set(physical)) == len(physical),
          "Dependency physical table names collide or are unsafe")
    all_schemas = {**base_schemas, **schemas}
    data = {name: json.loads(files[("dependencies/" if name in tables else "") + "data/" + name + ".json"])
            for name in all_specs}
    for name, spec in tables.items():
        rows = data[name]
        check(len(rows) == spec["rows"] and all(set(row) == set(schemas[name]) for row in rows),
              "Dependency row/column mismatch: " + name)
        keys = [row[spec["key"]] for row in rows]
        check(all(value is not None and value != "" for row in rows for value in row.values())
              and len(set(keys)) == len(keys), "Dependency values/keys missing or duplicated: " + name)
        for column, kind in schemas[name].items():
            if kind == "int64":
                check(all(type(row[column]) is int and row[column] >= 0 for row in rows),
                      "Invalid dependency quantity: " + name + "." + column)
    for relationship in contract["relationships"]:
        source, target, column = relationship["from"], relationship["to"], relationship["column"]
        check(source in tables and target in all_specs and column in all_schemas[source],
              "Invalid dependency relationship")
        target_key = all_specs[target]["key"]
        keys = {row[target_key] for row in data[target]}
        check(all(row[column] in keys for row in data[source]),
              "Dependency foreign key missing: " + relationship["name"])
    for name, columns in (("BillOfMaterial", ("ProductId", "ComponentId")),
                          ("ComponentSupply", ("ComponentId", "SupplierSiteId")),
                          ("RouteHub", ("SupplyRouteId", "LogisticsHubId"))):
        pairs = [tuple(row[column] for column in columns) for row in data[name]]
        check(len(set(pairs)) == len(pairs), "Duplicate dependency association: " + name)
    routes = {row["SupplyRouteId"]: row for row in data["SupplyRoute"]}
    check(all(row["SupplierSiteId"] == routes[row["SupplyRouteId"]]["SupplierSiteId"]
              and row["LeadTimeHours"] >= routes[row["SupplyRouteId"]]["TransitHours"]
              and row["DailyCapacityUnits"] > 0 for row in data["ComponentSupply"]),
          "Component supply site/route/lead-time mismatch")
    for identifier in routes:
        stops = [row["StopSequence"] for row in data["RouteHub"] if row["SupplyRouteId"] == identifier]
        check(stops and sorted(stops) == list(range(1, len(stops) + 1)),
              "Route requires unique contiguous hub stops")
    check(all(row["UnitsPerProduct"] > 0 for row in data["BillOfMaterial"]),
          "Bill-of-material quantity must be positive")
    check(all(row["SnapshotTime"] == contract["asOfUtc"] for row in data["LogisticsHub"]),
          "Dependency snapshot must match the workshop checkpoint")
    check(all(row["Status"] in ("Open", "Closed")
              and ((row["Status"] == "Open" and row["ExpectedRecoveryHours"] == 0)
                   or (row["Status"] == "Closed" and row["ExpectedRecoveryHours"] > 0))
              for row in data["LogisticsHub"]), "Unknown or inconsistent hub availability")
    return contract, schemas


def wait_for_standard_refresh(client, history_url, request_id, timeout=900):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        history = client.get(history_url)
        matching = [r for r in history["value"] if r.get("requestId") == request_id]
        check(len(matching) <= 1, "Ambiguous refresh request in history")
        if matching:
            status = matching[0]["status"]
            if status == "Completed":
                return
            check(status not in ("Failed", "Cancelled", "Disabled"), "Model refresh failed")
            check(status in ("Unknown", "NotStarted", "InProgress"),
                  "Unrecognized model refresh state; inspect the exact request")
        time.sleep(10)
    raise TimeoutError("Exact model refresh is still pending/not visible; do not submit another refresh")


def standard_refresh_request_id(location, request_id, dataset_url):
    if location:
        parsed = urllib.parse.urlsplit(location)
        host = parsed.hostname or ""
        expected_path = urllib.parse.urlsplit(dataset_url).path + "/refreshes/"
        check(parsed.scheme == "https" and parsed.port in (None, 443)
              and not parsed.username and not parsed.password and not parsed.query and not parsed.fragment
              and (host == "api.powerbi.com"
                   or host.endswith((".analysis.windows.net", ".msitapi.fabric.microsoft.com")))
              and parsed.path.startswith(expected_path), "Unexpected refresh Location; inspect before retry")
        location_id = uuid_text(parsed.path[len(expected_path):])
        check(not request_id or uuid_text(request_id) == location_id, "Refresh identifiers disagree")
        request_id = location_id
    check(request_id, "Refresh request ID missing; inspect history before retrying")
    return uuid_text(request_id)


class Stager:
    def __init__(self, utilities, spark_session, files, bundle_hash, provenance=None):
        self.utils, self.spark, self.files = utilities, spark_session, files
        self.contract, self.schemas = validate_inputs(files)
        self.dependencies, dependency_schemas = validate_dependencies(files, self.contract, self.schemas)
        self.tables = {**self.contract["tables"], **self.dependencies.get("tables", {})}
        self.schemas = {**self.schemas, **dependency_schemas}
        dependency_files = {name: text for name, text in files.items() if name.startswith("dependencies/")}
        self.dependency_hash = (hashlib.sha256(json.dumps(
            dependency_files, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
            if dependency_files else None)
        self.source_specs = {key: dict(spec) for key, spec in self.contract["sources"].items()}
        self.context = utilities.runtime.context
        self.workspace_id = uuid_text(self.context["currentWorkspaceId"])
        self.notebook_id = uuid_text(self.context["currentNotebookId"])
        self.base = FABRIC_API + "/v1/workspaces/" + self.workspace_id
        self.client = FabricClient(utilities)
        self.bundle_hash = bundle_hash
        self.provenance = provenance
        self.marker = "Ignite2026 starter; notebook=" + self.notebook_id + "; bundle=" + bundle_hash
        self.state, self.state_path, self.files_uri = None, None, None

    def write_state(self):
        self.state["updatedAtUtc"] = timestamp()
        check(self.utils.fs.put(self.state_path, json.dumps(self.state, indent=2), True),
              "Could not persist deployment state; stop before further mutations")

    def select_source_names(self, items):
        suffix = "_Lab_" + self.notebook_id.replace("-", "")[:12]
        base_name = self.source_specs["lakehouse"]["name"]
        owned_names = {i["displayName"] for i in items
                       if i["type"] == "Lakehouse" and i.get("description") == self.marker}
        owns_base, owns_scoped = base_name in owned_names, base_name + suffix in owned_names
        check(not (owns_base and owns_scoped), "Ambiguous prior deployment; inspect both owned Lakehouses")
        use_suffix = owns_scoped or (self.notebook_folder_id is not None and not owns_base)
        for key, spec in self.source_specs.items():
            self.contract["sources"][key] = {**spec, "name": spec["name"] + (suffix if use_suffix else "")}

    def preflight(self, confirmed_name):
        workspace = self.client.get(self.base)
        self.workspace_name = workspace["displayName"]
        check(workspace["id"] == self.workspace_id and workspace.get("capacityId"),
              "Current workspace has no assigned Fabric capacity")
        check(not self.context.get("defaultLakehouseId"),
              "Detach the default lakehouse and restart the session before staging")
        matching = [w for w in self.client.listing(FABRIC_API + "/v1/workspaces")
                    if w["displayName"] == self.workspace_name]
        check(len(matching) == 1 and matching[0]["id"] == self.workspace_id,
              "Ambiguous accessible workspace name; four-part Spark names would be unsafe")
        items = self.client.listing(self.base + "/items")
        folders = self.client.listing(self.base + "/folders")
        notebooks = [i for i in items if i["id"] == self.notebook_id and i["type"] == "Notebook"]
        check(len(notebooks) == 1 and notebooks[0]["workspaceId"] == self.workspace_id,
              "Cannot identify this imported notebook")
        folder_id = notebooks[0].get("folderId")
        self.notebook_folder_id = uuid_text(folder_id) if folder_id is not None else None
        self.deployment_folder = None
        if self.notebook_folder_id is not None:
            self.deployment_folder = self.client.get(self.base + "/folders/" + self.notebook_folder_id)
            check(self.deployment_folder["id"] == self.notebook_folder_id
                  and self.deployment_folder["workspaceId"] == self.workspace_id,
                  "Notebook folder identity/workspace mismatch")
        self.select_source_names(items)
        print("Target:", self.workspace_name)
        if self.deployment_folder:
            print("Target folder:", self.deployment_folder["displayName"], "(" + self.notebook_folder_id + ")")
        else:
            print("Target folder:", self.contract["folderName"], "(created at workspace root on first apply)")
        print("Stages:", len(self.tables), "Delta tables (" + str(len(self.dependencies.get("tables", {})))
              + " Lakehouse-only), source Eventhouse, four telemetry tables, unchanged 13-table semantic model.")
        print("No completed ontology, Graph, dashboard or agent is deployed.")
        for key, spec in self.contract["sources"].items():
            print("Source " + key + ":", spec["name"])
        if confirmed_name is not None:
            check(confirmed_name == self.workspace_name, "Type the exact current workspace name to apply")
        name = self.contract["sources"]["lakehouse"]["name"]
        matches = [i for i in items if i["type"] == "Lakehouse" and i["displayName"].casefold() == name.casefold()]
        if matches:
            check(len(matches) == 1 and matches[0].get("description") == self.marker,
                  "Existing Lakehouse is not owned by this notebook and bundle: " + name
                  + ". No adoption or overwrite; inspect the conflicting item before retrying.")
            check(self.notebook_folder_id is None or matches[0].get("folderId") == self.notebook_folder_id,
                  "Owned Lakehouse is outside the notebook folder; do not move the notebook after deployment")
            lakehouse = self.client.get(self.base + "/lakehouses/" + matches[0]["id"])
            self.files_uri, self.one_lake_root = one_lake_files(
                lakehouse["properties"], self.workspace_id, lakehouse["id"])
            self.state_path = self.files_uri + "/ignite2026/bootstrap-state.json"
            check(self.utils.fs.exists(self.state_path),
                  "Owned Lakehouse has no ledger. Interrupted initial creation needs manual inspection.")
            self.state = json.loads(self.utils.fs.head(self.state_path, 1024 * 1024))
            check(self.state["workspaceId"] == self.workspace_id
                  and self.state["notebookId"] == self.notebook_id
                  and self.state["bundleHash"] == self.bundle_hash
                  and self.state["items"]["lakehouse"] == lakehouse["id"], "Deployment ledger mismatch")
            check(self.state.get("notebookFolderId") == self.notebook_folder_id
                  and (self.notebook_folder_id is None or self.state["folderId"] == self.notebook_folder_id),
                  "Notebook folder changed since deployment; restore its original location before resuming")
            check("sourceNames" not in self.state
                  or self.state["sourceNames"] == {k: v["name"] for k, v in self.contract["sources"].items()},
                  "Deployment source-name mismatch; inspect the existing ledger")
            self.owned_item("lakehouse")
            check(self.state.get("dependencyInputsHash") in (None, self.dependency_hash),
                  "Dependency inputs changed after staging; no overwrite or automatic data migration")
            pending = self.state.get("pending")
            if pending and confirmed_name is not None and pending.get("kind") == "create":
                self.reconcile_pending_create()
            elif pending and confirmed_name is not None and pending.get("kind") == "modelRefresh":
                self.reconcile_pending_refresh()
            check(not self.state.get("pending"), "Uncertain prior write. Inspect the ledger; do not replay it.")
        else:
            if self.notebook_folder_id is None:
                check(len(items) == 1 and items[0]["id"] == self.notebook_id and not folders,
                      "First deployment requires a completely blank workspace except for this notebook, "
                      "or place the notebook in a deployment folder")
            for spec in self.contract["sources"].values():
                check(not any(i["type"] == spec["type"]
                              and i["displayName"].casefold() == spec["name"].casefold() for i in items),
                      "Existing source name conflicts with deployment; no adoption or overwrite: " + spec["name"])
        return workspace

    def owned_item(self, key):
        spec = self.contract["sources"][key]
        item = self.client.get(self.base + "/items/" + uuid_text(self.state["items"][key]))
        check(item["workspaceId"] == self.workspace_id and item["type"] == spec["type"]
              and item["displayName"] == spec["name"] and item.get("description") == self.marker
              and item.get("folderId") == self.state["folderId"], "Owned item identity/scope drift: " + key)
        return item

    def remember_operation(self, location):
        self.state["pending"]["operationLocation"] = location
        self.write_state()

    def reconcile_pending_create(self):
        pending = self.state["pending"]
        key = pending.get("key")
        check(pending.get("kind") == "create" and key in self.contract["sources"]
              and key not in self.state["items"] and pending.get("operationLocation"),
              "Pending create has no exact recoverable operation; inspect the ledger")
        operation_url = fabric_operation_url(pending["operationLocation"])
        result = self.client.finish_operation(operation_url)
        identifier = uuid_text(result["id"])
        spec = self.contract["sources"][key]
        item = self.client.get(self.base + "/items/" + identifier)
        check(item["id"] == identifier and item["workspaceId"] == self.workspace_id
              and item["type"] == spec["type"] and item["displayName"] == spec["name"]
              and item.get("description") == self.marker and item.get("folderId") == self.state["folderId"],
              "Completed create result is not the expected owned item; pending ledger retained")
        if key == "database":
            database = self.client.get(self.base + "/kqlDatabases/" + identifier)
            check(database["properties"]["parentEventhouseItemId"] == self.state["items"].get("eventhouse"),
                  "Completed database create has an unexpected parent; pending ledger retained")
        self.state["items"][key] = identifier
        self.state.setdefault("reconciledCreates", []).append({
            "key": key, "operationUrl": operation_url, "itemId": identifier, "completedAtUtc": timestamp()})
        self.state["pending"] = None
        self.write_state()
        print("Recovered completed create from the saved operation:", key)

    def reconcile_pending_refresh(self):
        pending = self.state["pending"]
        self.owned_item("model")
        history_url = (POWERBI_API + "/v1.0/myorg/groups/" + self.workspace_id + "/datasets/"
                       + self.state["items"]["model"] + "/refreshes?$top=60")
        check(pending.get("refreshRequestId") and pending.get("refreshHistoryUrl") == history_url,
              "Pending refresh has no verified request ID/history URL; inspect history before recovery")
        request_id = uuid_text(pending["refreshRequestId"])
        wait_for_standard_refresh(self.client, history_url, request_id)
        self.state["steps"]["modelRefresh"] = "verified"
        self.state.setdefault("reconciledRefreshes", []).append({
            "requestId": request_id, "completedAtUtc": timestamp()})
        self.state["pending"] = None
        self.write_state()
        print("Recovered completed model refresh from its saved request ID")

    def provision(self):
        if self.state is not None:
            folder = self.client.get(self.base + "/folders/" + self.state["folderId"])
            check(folder["id"] == self.state["folderId"]
                  and folder["displayName"] == self.state.get("folderName", self.contract["folderName"])
                  and folder["workspaceId"] == self.workspace_id, "Deployment folder drift")
            return
        folder = self.deployment_folder or self.client.create(
            self.base + "/folders", {"displayName": self.contract["folderName"]})
        spec = self.contract["sources"]["lakehouse"]
        lakehouse = self.client.create(self.base + "/lakehouses", {
            "displayName": spec["name"], "description": self.marker, "folderId": folder["id"],
            "creationPayload": {"enableSchemas": True}})
        deadline = time.monotonic() + 600
        while True:
            detail = self.client.get(self.base + "/lakehouses/" + lakehouse["id"])
            if detail.get("properties", {}).get("oneLakeFilesPath"):
                break
            check(time.monotonic() < deadline, "OneLake properties not ready; inspect initial provisioning")
            time.sleep(10)
        self.files_uri, self.one_lake_root = one_lake_files(
            detail["properties"], self.workspace_id, lakehouse["id"])
        self.state_path = self.files_uri + "/ignite2026/bootstrap-state.json"
        self.state = {"workspaceId": self.workspace_id, "notebookId": self.notebook_id,
                      "bundleHash": self.bundle_hash, "folderId": folder["id"],
                      "notebookFolderId": self.notebook_folder_id, "folderName": folder["displayName"],
                      "sourceNames": {key: spec["name"] for key, spec in self.contract["sources"].items()},
                      "items": {"lakehouse": lakehouse["id"]}, "steps": {}, "pending": None}
        self.utils.fs.mkdirs(self.files_uri + "/ignite2026")
        self.write_state()
        self.owned_item("lakehouse")

    def create_item(self, key, extra=None):
        if key in self.state["items"]:
            return self.owned_item(key)
        spec = self.contract["sources"][key]
        self.state["pending"] = {"kind": "create", "key": key, "startedAtUtc": timestamp()}
        self.write_state()
        body = {"displayName": spec["name"], "description": self.marker, "folderId": self.state["folderId"]}
        body.update(extra or {})
        result = self.client.create(self.base + "/" + spec["route"], body, self.remember_operation)
        self.state["items"][key] = uuid_text(result["id"])
        self.state["pending"] = None
        self.write_state()
        return self.owned_item(key)

    def delta_tables(self):
        from pyspark.sql import functions as F
        self.owned_item("lakehouse")
        self.spark.conf.set("spark.sql.session.timeZone", "UTC")
        types = {"string": "string", "int64": "long", "double": "double",
                 "decimal": "decimal(38,18)", "dateTime": "timestamp", "boolean": "boolean"}
        for name, spec in self.tables.items():
            prefix = "dependencies/" if name in self.dependencies.get("tables", {}) else ""
            schema = self.schemas[name]
            frame = self.spark.read.option("multiLine", "true").schema(
                ", ".join("`" + c + "` string" for c in schema)).json(
                    self.files_uri + "/ignite2026/input/" + prefix + "data/" + name + ".json")
            frame = frame.select(*(F.col(c).cast(types[t]).alias(c) for c, t in schema.items()))
            check(not frame.where(" OR ".join("`" + c + "` IS NULL" for c in schema)).take(1),
                  "Unexpected null/cast failure in fixture: " + name)
            identifier = table_identifier(self.workspace_name, self.contract["sources"]["lakehouse"]["name"],
                                          spec["physical"])
            if not self.spark.catalog.tableExists(identifier):
                frame.write.format("delta").mode("errorifexists").saveAsTable(identifier)
            detail = self.spark.sql("DESCRIBE DETAIL " + identifier).first().asDict()
            location = urllib.parse.urlsplit(detail["location"])
            check(location.scheme == "abfss" and location.hostname == self.files_uri.split("@", 1)[1].split("/", 1)[0]
                  and location.username == self.workspace_id
                  and location.path == "/" + self.state["items"]["lakehouse"] + "/Tables/dbo/" + spec["physical"],
                  "Resolved Delta table is outside the owned Lakehouse")
            actual = self.spark.table(identifier).select(*schema)
            check(actual.count() == spec["rows"] and not actual.exceptAll(frame).take(1)
                  and not frame.exceptAll(actual).take(1), "Existing table differs; never overwrite: " + name)
            self.state["steps"]["delta:" + name] = "verified"
            self.write_state()
            print("Delta verified:", name, spec["rows"], "rows")
    def stage_inputs(self):
        for name, text in self.files.items():
            parent = name.rsplit("/", 1)[0] if "/" in name else ""
            self.utils.fs.mkdirs(self.files_uri + "/ignite2026/input/" + parent)
            check(self.utils.fs.put(self.files_uri + "/ignite2026/input/" + name, text, True),
                  "Could not persist input file: " + name)

    def kusto(self, query, management=False):
        uri = self.database["properties"]["queryServiceUri"].rstrip("/")
        _, _, body = self.client.call("POST", uri + ("/v1/rest/mgmt" if management else "/v1/rest/query"),
                                      {"db": self.database["displayName"], "csl": query}, "kusto")
        return kql_result(body)

    def eventhouse_databases(self, eventhouse):
        detail = self.client.get(self.base + "/eventhouses/" + uuid_text(eventhouse["id"]))
        spec = self.contract["sources"]["eventhouse"]
        check(detail["id"] == eventhouse["id"] and detail["workspaceId"] == self.workspace_id
              and detail["type"] == "Eventhouse" and detail["displayName"] == spec["name"]
              and detail.get("description") == self.marker
              and detail.get("folderId") == self.state["folderId"], "Owned Eventhouse scope drift")
        identifiers = detail.get("properties", {}).get("databasesItemIds")
        check(isinstance(identifiers, list),
              "Eventhouse child metadata is not ready; resume this same notebook later, not a new import")
        identifiers = [uuid_text(identifier) for identifier in identifiers]
        check(len(identifiers) == len(set(identifiers)), "Duplicate Eventhouse database child IDs")
        databases = []
        for identifier in identifiers:
            database = self.client.get(self.base + "/kqlDatabases/" + identifier)
            check(database["id"] == identifier and database["type"] == "KQLDatabase"
                  and database["workspaceId"] == self.workspace_id
                  and database["properties"]["parentEventhouseItemId"] == eventhouse["id"]
                  and database.get("folderId") == self.state["folderId"],
                  "Eventhouse database child identity/parent/folder mismatch")
            databases.append(database)
        return databases

    def telemetry(self):
        eventhouse = self.create_item("eventhouse")
        if "database" not in self.state["items"]:
            # Accept only an auto-generated database whose parent is our newly owned Eventhouse.
            candidates = self.eventhouse_databases(eventhouse)
            generated = [i for i in candidates if i["displayName"] == self.contract["sources"]["database"]["name"]]
            if generated:
                check(len(generated) == 1, "Ambiguous automatic KQL database")
                detail = generated[0]
                check(detail["properties"]["parentEventhouseItemId"] == eventhouse["id"]
                      and detail["workspaceId"] == self.workspace_id
                      and detail.get("folderId") == self.state["folderId"],
                      "Existing database is not an in-folder child of the owned Eventhouse")
                # A child inherits provenance from its owned parent, not a guessed name.
                self.state["items"]["database"] = generated[0]["id"]
                self.state["generatedDatabase"] = True
                self.write_state()
            else:
                self.create_item("database", {"creationPayload": {
                    "databaseType": "ReadWrite", "parentEventhouseItemId": eventhouse["id"]}})
        if not self.state.get("generatedDatabase"):
            self.owned_item("database")
        self.database = self.client.get(self.base + "/kqlDatabases/" + self.state["items"]["database"])
        check(self.database["properties"]["parentEventhouseItemId"] == eventhouse["id"]
              and self.database["workspaceId"] == self.workspace_id
              and self.database["displayName"] == self.contract["sources"]["database"]["name"]
              and self.database.get("folderId") == self.state["folderId"], "KQL parent/scope drift")
        for table, schema in json.loads(self.files["telemetry-schemas.json"]).items():
            rows = json.loads(self.files["telemetry/" + table + ".json"])
            check(re.fullmatch(r"[A-Za-z]+", table), "Unexpected telemetry table name")
            declaration = ",".join(c + ":" + t for c, t in schema.items())
            self.kusto(".create-merge table " + table + " (" + declaration + ")", True)
            step = "telemetry:" + table
            if self.state["steps"].get(step) not in ("verified", "ingested"):
                expressions = []
                for row in rows:
                    expressions.extend("datetime(" + row[c] + ")" if t == "datetime" else json.dumps(row[c])
                                       for c, t in schema.items())
                query = ".set-or-append " + table + " <| datatable(" + declaration + ")[" + ",".join(expressions) + "]"
                self.state["pending"] = {"kind": "ingest", "table": table, "startedAtUtc": timestamp()}
                self.write_state()
                self.kusto(query, True)
                self.state["steps"][step] = "ingested"
                self.state["pending"] = None
                self.write_state()
            actual = self.kusto(table + " | where ObservationTime between "
                                "(datetime(2027-12-31T00:00:00Z) .. datetime(2027-12-31T20:00:00Z))"
                                " | project " + ",".join(schema))
            def canonical(data):
                normalized = []
                for row in data:
                    values = dict(row)
                    for column, kind in schema.items():
                        if kind == "real":
                            values[column] = float(row[column])
                        elif kind == "datetime":
                            values[column] = datetime.fromisoformat(row[column].replace("Z", "+00:00")).isoformat()
                    normalized.append(json.dumps(values, sort_keys=True))
                return sorted(normalized)
            check(canonical(actual) == canonical(rows), "Telemetry verification mismatch; do not append again")
            self.state["steps"][step] = "verified"
            self.write_state()
            print("Telemetry verified:", table, len(rows), "rows")

    def semantic_model(self):
        self.create_item("model", {"definition": render_model(self.files, self.one_lake_root)})
        if self.state["steps"].get("modelRefresh") == "verified":
            return
        url = POWERBI_API + "/v1.0/myorg/groups/" + self.workspace_id + "/datasets/" + self.state["items"]["model"]
        self.state["pending"] = {"kind": "modelRefresh", "startedAtUtc": timestamp()}
        self.write_state()
        status, headers, _ = self.client.call("POST", url + "/refreshes", {"notifyOption": "NoNotification"})
        check(status == 202, "Model framing/refresh not accepted")
        location = headers.get("location", "")
        request_id = headers.get("x-ms-request-id") or headers.get("requestid")
        self.state["pending"]["refreshResponseHeaders"] = {"location": location, "requestId": request_id}
        self.write_state()
        request_id = standard_refresh_request_id(location, request_id, url)
        history_url = url + "/refreshes?$top=60"
        self.state["pending"]["refreshRequestId"] = request_id
        self.state["pending"]["refreshHistoryUrl"] = history_url
        self.write_state()
        # Standard refreshes are tracked in history, not enhanced-refresh execution details.
        wait_for_standard_refresh(self.client, history_url, request_id)
        self.state["steps"]["modelRefresh"] = "verified"
        self.state["pending"] = None
        self.write_state()

    def run(self, apply=False, confirmed_name=""):
        self.preflight(confirmed_name if apply else None)
        if not apply:
            print("PREVIEW ONLY. Set APPLY=True and CONFIRM_WORKSPACE_NAME to the name above, then Run all.")
            return {"status": "preview", "workspaceName": self.workspace_name,
                    "folderId": self.notebook_folder_id,
                    "folderName": (self.deployment_folder["displayName"] if self.deployment_folder
                                   else self.contract["folderName"]),
                    "sourceNames": {k: v["name"] for k, v in self.contract["sources"].items()}}
        self.provision()
        try:
            if self.provenance:
                self.state["source"] = self.provenance
                self.write_state()
            if self.dependency_hash:
                self.state["dependencyInputsHash"] = self.dependency_hash
                self.write_state()
            self.stage_inputs()
            self.delta_tables()
            self.telemetry()
            self.semantic_model()
            self.state["status"] = "sources staged; semantic model refresh completed; DAX smoke check still required"
            self.state["participantOntology"] = "not created"
            self.state.pop("errorType", None)
            self.write_state()
        except Exception as error:
            self.state["status"] = "failed"
            self.state["errorType"] = type(error).__name__
            self.write_state()
            raise
        print(self.state["status"])
        print("Next: open the semantic model, check one measure, then Generate Ontology.")
        if self.dependencies:
            print("Lakehouse-only entity tables:", ", ".join(self.dependencies["tables"]))
            print("Discover and add these through the ontology; they are deliberately absent from the semantic model.")
        print("State and input files:", self.files_uri + "/ignite2026")
        return {"status": self.state["status"], "items": self.state["items"],
                "folderId": self.state["folderId"],
                "sourceNames": {k: v["name"] for k, v in self.contract["sources"].items()}}


## Download and validate inputs
Resolve one commit, verify every file's length and SHA-256, and validate keys/joins before provisioning. A failed download stops setup; there is no embedded-data fallback.


In [ ]:
files, bundle_hash, source_revision = download_inputs(REPO_REVISION)


In [ ]:
import notebookutils
deployment = Stager(notebookutils, spark, files, bundle_hash, provenance=source_revision)
result = deployment.run(apply=APPLY, confirmed_name=CONFIRM_WORKSPACE_NAME)
